# Archive Exploration

Bonanza Creek LTER assembled this data to quantify population dynamics of snowshoe hares (_Lepus americanus_) in northern boreal forests from 1999-2012. One row in the data represents one observation in the capture-recapture study. Hares and sessions were given unique id numbers as to avoid overcounting overzealous participants. Their weights were measured in grams. The metadata does not reveal how missing data was encoded and there is no sensitive information in the dataset.


![Snowshoe Hare. [Source: Wikimedia Commons](Creative Commons Attribution-ShareAlike License)](https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1920px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20170313021652)


Kielland, K., F.S. Chapin, R.W. Ruess, and Bonanza Creek LTER. 2017. Snowshoe hare physical data in Bonanza Creek Experimental Forest: 1999-Present ver 22. Environmental Data Initiative. https://doi.org/10.6073/pasta/03dce4856d79b91557d8e6ce2cbcdc14 (Accessed 2026-10-15).

# Data loading and preliminary exploration

In [46]:
import pandas as pd
import numpy as np
hares = pd.read_csv('data/55_Hare_Data_2012.txt')

In [30]:
hares.shape

(3380, 14)

In [31]:
hares.columns

Index(['date', 'time', 'grid', 'trap', 'l_ear', 'r_ear', 'sex', 'age',
       'weight', 'hindft', 'notes', 'b_key', 'session_id', 'study'],
      dtype='object')

In [32]:
hares.describe()

,weight,hindft,b_key,session_id
count,2845.000000,1633.000000,3333.000000,3380.000000
mean,1346.081547,130.872627,500.640864,53.232249
std,345.160112,16.155295,299.421121,33.171355
min,0.000000,60.000000,1.000000,1.000000
25%,1180.000000,128.000000,235.000000,22.000000
50%,1400.000000,135.000000,464.000000,55.000000
75%,1580.000000,140.000000,755.000000,82.000000
max,2365.000000,160.000000,1034.000000,113.000000


In [33]:
hares.dtypes

date           object
time           object
grid           object
trap           object
l_ear          object
r_ear          object
sex            object
age            object
weight        float64
hindft        float64
notes          object
b_key         float64
session_id      int64
study          object
dtype: object

In [34]:
hares.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3380 entries, 0 to 3379
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        3380 non-null   object 
 1   time        264 non-null    object 
 2   grid        3380 non-null   object 
 3   trap        3368 non-null   object 
 4   l_ear       3332 non-null   object 
 5   r_ear       3211 non-null   object 
 6   sex         3028 non-null   object 
 7   age         1269 non-null   object 
 8   weight      2845 non-null   float64
 9   hindft      1633 non-null   float64
 10  notes       243 non-null    object 
 11  b_key       3333 non-null   float64
 12  session_id  3380 non-null   int64  
 13  study       3217 non-null   object 
dtypes: float64(3), int64(1), object(10)
memory usage: 369.8+ KB


In [35]:
print(hares['weight'].min())
print(hares['weight'].max())

0.0
2365.0


In [36]:
print(hares['hindft'].min())
print(hares['hindft'].max())

60.0
160.0


In [37]:
hares['trap'].value_counts()

trap
3e     75
1c     65
1e     64
2e     61
4c     60
       ..
1e      1
6b      1
2c      1
5c      1
4b      1
Name: count, Length: 121, dtype: int64

In [38]:
hares['grid'].value_counts()

grid
bonrip     2030
bonbs       929
bonmat      260
Bonbs       152
bonmat        9
Name: count, dtype: int64

# Detecting messy values

| value  | meaning |
| --- | ----------- |
| m | male |
| f | female |
| m? | male not confirmed

In [57]:
hares['sex'].value_counts(dropna=False)

sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64

In [58]:
hares['sex'].nunique()

12

No, the values in the metadata do not match with the data. It is likeley that it is difficult to identify sex of hares when they are young, or in the dark. Yes, there appears to be repeated values like. It is likeley that there are extra spaces/ 

In [59]:
hares['sex'].str.strip().nunique()

10

In [60]:
sex_hares = hares['sex'].str.strip().str.lower()
sex_hares.value_counts()

# sex_hares[(sex_hares == '?') |]

sex
f     1721
m     1249
?       40
f?      13
m?       4
pf       1
Name: count, dtype: int64

In [64]:
sex_hares_fm = sex_hares.replace('?', np.nan).replace('f?', np.nan).replace('m?', np.nan).replace('pf', np.nan)
sex_hares_fm.value_counts(dropna=False)

sex
f      1721
m      1249
NaN     410
Name: count, dtype: int64

In [63]:
sex_hares.value_counts()

sex
f     1721
m     1249
?       40
f?      13
m?       4
pf       1
Name: count, dtype: int64

In [ ]:
sex_simple = sex_hares.replace('?', np.nan).replace('pf', np.nan).replace('f?', np.nan)
sex_simple.value_counts(dropna=False)

sex
f      1721
m      1249
NaN     406
m?        4
Name: count, dtype: int64

Kept m, f, m? to match the meta data

# Calculate mean weight

In [73]:
hares['sex'] = sex_simple

hares['sex'].value_counts()

sex
f     1721
m     1249
m?       4
Name: count, dtype: int64

In [75]:
hares.groupby('sex').mean('weight')

,weight,hindft,b_key,session_id
sex,,,,
f,1365.164792,130.878889,481.653263,57.746078
m,1349.935542,133.194074,489.514812,49.230584
m?,1446.666667,134.250000,568.000000,48.750000


Female snowshoe hares are, on average, heavier (~ 1365.2 grams) compared to male snowshoe hares (~ 1349.9 grams). Though, hares believed to be male are heavier, than both female and male hares. 

# Compile code

In [81]:
sex_simple = hares['sex'].replace('?', np.nan).replace('f?', np.nan).replace('pf', np.nan)
hares['sex'] = sex_simple
hares.groupby('sex').mean('weight')


,weight,hindft,b_key,session_id
sex,,,,
f,1365.164792,130.878889,481.653263,57.746078
m,1349.935542,133.194074,489.514812,49.230584
m?,1446.666667,134.250000,568.000000,48.750000
